In [1]:
import os
import shutil
import json
import torch
import time
import nltk
import numpy as np
from google.colab import drive
from transformers import BartTokenizer, BartForConditionalGeneration
from datasets import Dataset

# ==========================================
# 1. CẤP QUYỀN CHO NUMPY (FIX LỖI UNPICKLING)
# ==========================================
try:
    reconstruct_func = np._core.multiarray._reconstruct if hasattr(np, '_core') else np.core.multiarray._reconstruct
    torch.serialization.add_safe_globals([
        reconstruct_func,
        np.ndarray,
        np.dtype,
    ])
    print("✅ Đã whitelist Numpy cho PyTorch.")
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override
print(f"✅ Đã ép buộc torch.load(weights_only=False) thành công.")

# ==========================================
# 2. KẾT NỐI DRIVE & TẢI DỮ LIỆU
# ==========================================
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')
FINAL_SAVE_PATH = "/content/drive/My Drive/BART_Large_Spider_CRP_FFT"

print(">>> [1/4] Kiểm tra và tải dữ liệu...")
if not os.path.exists('spider_data'):
    os.system("pip install -q kaggle")
    os.system("kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset")
    os.system("unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider")
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
        shutil.rmtree('temp_spider')
        os.system("wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py")
        os.system("wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py")
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)

# Tải bổ sung Spider-Realistic
if not os.path.exists('spider_data/spider-realistic.json'):
    os.system("wget -q -O spider_data/spider-realistic.json \"https://zenodo.org/records/5205322/files/spider-realistic.json?download=1\"")

# ==========================================
# 3. TIỀN XỬ LÝ SCHEMA VÀ LOAD DATASET
# ==========================================
with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[0]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

print(">>> [2/4] Đang load tập Spider-Realistic...")
val_ds = load_spider_unified("spider_data/spider-realistic.json")

# ==========================================
# 4. LOAD MÔ HÌNH TỪ GOOGLE DRIVE
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n>>> [3/4] Đang tải mô hình BART-Large từ {FINAL_SAVE_PATH} lên {device}...")

tokenizer = BartTokenizer.from_pretrained(FINAL_SAVE_PATH)
model = BartForConditionalGeneration.from_pretrained(FINAL_SAVE_PATH).to(device)
model.eval()

# ==========================================
# 5. CHẠY INFERENCE VÀ ĐÁNH GIÁ
# ==========================================
print("\n>>> [4/4] Bắt đầu chạy dự đoán trên tập Spider-Realistic (508 câu)...")
predictions, gold_lines = [], []

start_time = time.time()
total_items = len(val_ds)

for idx, item in enumerate(val_ds):
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

    if (idx + 1) % 50 == 0 or (idx + 1) == total_items:
        print(f"Đã dự đoán: {idx + 1}/{total_items} câu")

print(f"\n✅ Hoàn tất dự đoán trong {time.time() - start_time:.2f} giây.")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

print("\n>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:")
os.system('sed -i \'s/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\\n    conn.text_factory = lambda b: b.decode(errors="ignore")/\' evaluation.py')
os.system('python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all')

✅ Đã whitelist Numpy cho PyTorch.
✅ Đã ép buộc torch.load(weights_only=False) thành công.
Mounted at /content/drive
>>> [1/4] Kiểm tra và tải dữ liệu...
>>> [2/4] Đang load tập Spider-Realistic...

>>> [3/4] Đang tải mô hình BART-Large từ /content/drive/My Drive/BART_Large_Spider_CRP_FFT lên cuda...


Loading weights:   0%|          | 0/514 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



>>> [4/4] Bắt đầu chạy dự đoán trên tập Spider-Realistic (508 câu)...
Đã dự đoán: 50/508 câu
Đã dự đoán: 100/508 câu
Đã dự đoán: 150/508 câu
Đã dự đoán: 200/508 câu
Đã dự đoán: 250/508 câu
Đã dự đoán: 300/508 câu
Đã dự đoán: 350/508 câu
Đã dự đoán: 400/508 câu
Đã dự đoán: 450/508 câu
Đã dự đoán: 500/508 câu
Đã dự đoán: 508/508 câu

✅ Hoàn tất dự đoán trong 310.92 giây.

>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:


0

In [3]:
print("\n>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all


>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:
eval_err_num:1
medium pred: SELECT avg(Age) ,  min(age) , ,  max FROM singer WHERE country  =  'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:2
medium pred: SELECT avg(Age) ,  min(age) , ,  max(AGE) FROM singer WHERE country  =  "France"
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  <  2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  <  2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

eval_err_num:3
extra pred: SELECT T2.name ,  T1.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.'Stadium_ID  =  T2' AND T1
extra gold: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadiu